In [23]:
import sys
sys.path.append('..')
from osp import *

In [24]:
df_meta = get_corpus_metadata()

In [25]:
df_probs = pd.read_pickle('../data/raw/df_probs2.pkl.gz')
id2disc = dict(zip(df_meta.index, df_meta['discipline']))
id2dec = dict(zip(df_meta.index, df_meta['decade']))
df_probs['true'] = df_probs['id'].apply(lambda x: id2disc[x.split('__')[0]])
df_probs['pred'] = df_probs['pred'].apply(lambda x: x.split()[-1])
df_probs['irrelev'] = [x not in y for x,y in zip(df_probs['true'], df_probs['comparison'])]
df_probs['same_group'] = [x.split()[1] == x.split()[-1] for x in df_probs['comparison']]
df_probs['same_period'] = [x.split()[0] == x.split()[-2] for x in df_probs['comparison']]
df_probs['correct'] = df_probs['pred'] == df_probs['true']
df_probs['redundant'] = [x.split(' vs ')[0] > x.split(' vs ')[1] for x in df_probs['comparison']]
df_probs['decade'] = df_probs['id'].apply(lambda x: id2dec[x.split('__')[0]])
# df_probs = df_probs[~df_probs.irrelev]
df_probs = df_probs[~df_probs.same_group]
df_probs = df_probs[df_probs.same_period]
# df_probs = df_probs[~df_probs.redundant]
# df_probs = df_probs[df_probs.comparison.str.contains('Philosophy')]
# df_probs = df_probs[df_probs.comparison.str.contains('Other')]
df_probs

,comparison,id,prob1,prob2,pred,true,irrelev,same_group,same_period,correct,redundant,decade
0,1900-1925 Literature vs 1900-1925 Other,lit/10.1086/341235__06,0.984598,0.015402,Literature,Literature,False,False,True,True,False,2000
1,1900-1925 Literature vs 1900-1925 Other,lit/10.1086/341236__02,0.060157,0.939843,Other,Literature,False,False,True,False,False,2000
2,1900-1925 Literature vs 1900-1925 Other,lit/10.1086/341236__03,0.197024,0.802976,Other,Literature,False,False,True,False,False,2000
3,1900-1925 Literature vs 1900-1925 Other,lit/10.1086/341237__04,0.506802,0.493198,Literature,Literature,False,False,True,True,False,2000
4,1900-1925 Literature vs 1900-1925 Other,lit/10.1086/341238__02,0.008694,0.991306,Other,Literature,False,False,True,False,False,2000
...,...,...,...,...,...,...,...,...,...,...,...,...
2894937,2000-2025 Philosophy vs 2000-2025 Other,phil/10.2307/48761521__05,0.625802,0.374198,Philosophy,Philosophy,False,False,True,True,True,2020
2894938,2000-2025 Philosophy vs 2000-2025 Other,phil/10.2307/48761523__01,0.998669,0.001331,Philosophy,Philosophy,False,False,True,True,True,2020
2894939,2000-2025 Philosophy vs 2000-2025 Other,phil/10.2307/48761525__02,0.902791,0.097209,Philosophy,Philosophy,False,False,True,True,True,2020
2894940,2000-2025 Philosophy vs 2000-2025 Other,phil/10.2307/48761525__03,0.996893,0.003107,Philosophy,Philosophy,False,False,True,True,True,2020


In [26]:
df_probs.groupby('comparison').correct.mean().sort_values()

comparison
1925-1950 Literature vs 1925-1950 Other         0.328420
1975-2000 Literature vs 1975-2000 Other         0.335466
1950-1975 Literature vs 1950-1975 Other         0.337798
1900-1925 Literature vs 1900-1925 Other         0.342343
2000-2025 Literature vs 2000-2025 Other         0.356347
1925-1950 Philosophy vs 1925-1950 Other         0.574352
1900-1925 Philosophy vs 1900-1925 Other         0.577011
2000-2025 Philosophy vs 2000-2025 Other         0.612694
1975-2000 Philosophy vs 1975-2000 Other         0.620389
1950-1975 Philosophy vs 1950-1975 Other         0.622280
1900-1925 Philosophy vs 1900-1925 Literature    0.747176
2000-2025 Philosophy vs 2000-2025 Literature    0.752824
1925-1950 Philosophy vs 1925-1950 Literature    0.753523
1975-2000 Philosophy vs 1975-2000 Literature    0.766477
1950-1975 Philosophy vs 1950-1975 Literature    0.783782
Name: correct, dtype: float64

In [27]:
figdf = get_avgs_df(df_probs, ['decade', 'true', 'comparison'], 'correct').reset_index().query('count>100')
figdf

,decade,true,comparison,mean,stderr,count
0,2020,Philosophy,1975-2000 Philosophy vs 1975-2000 Literature,0.974319,0.003622,1908
1,2020,Philosophy,1950-1975 Philosophy vs 1950-1975 Literature,0.969602,0.003931,1908
2,2010,Philosophy,1950-1975 Philosophy vs 1950-1975 Literature,0.966505,0.002671,4538
3,2010,Philosophy,1975-2000 Philosophy vs 1975-2000 Literature,0.964081,0.002763,4538
4,2020,Philosophy,1925-1950 Philosophy vs 1925-1950 Literature,0.961216,0.004421,1908
...,...,...,...,...,...,...
565,1980,Philosophy,1900-1925 Literature vs 1900-1925 Other,0.000000,0.000000,2278
566,1920,Philosophy,2000-2025 Literature vs 2000-2025 Other,0.000000,0.000000,703
567,1980,Philosophy,1925-1950 Literature vs 1925-1950 Other,0.000000,0.000000,2278
568,1980,Philosophy,1950-1975 Literature vs 1950-1975 Other,0.000000,0.000000,2278


In [ ]:
fig = (
    p9.ggplot(
        figdf,
        p9.aes(x='decade', y='mean',color='true')
    )
    + p9.geom_line()
    + p9.geom_errorbar(p9.aes(ymin='mean-stderr', ymax='mean+stderr'), width=0.2)
    + p9.facet_wrap('~comparison', ncol=4)
    + p9.theme_minimal()
    + p9.theme(figure_size=(15, 12))
    + p9.scale_y_continuous(limits=(.5,1))
)
fig